In [1]:
import torch
import numpy as np
from sklearn.datasets import load_diabetes, load_iris
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import r2_score, root_mean_squared_error, mean_squared_error
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

from torch import Tensor
from torch.nn import Sequential, Linear, ReLU, MSELoss, CrossEntropyLoss
from torch.utils.data import DataLoader, Dataset
from torch.optim import Adam

In [2]:
class MyDataset(Dataset):
    def __init__(self, datas: np.ndarray, labels: np.ndarray) -> None:
        self._x = torch.from_numpy(datas).float()
        self._y = torch.from_numpy(labels).float()

    def __len__(self) -> int:
        return len(self._x)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        int(idx)
        return self._x[idx], self._y[idx]

In [3]:
def train_model_reg(features, targets, model, skf, batch, epochs, optmizer, criterion):
    accuracy_scores = []
    f1_scores = []
    conf_matrix = []
    for i, (train_index, test_index) in enumerate(skf.split(features, targets)):
        print(f"Treinando modelo ----- SKFold {i + 1}")
        train_data = MyDataset(features[train_index], targets[train_index])
        train_loader = DataLoader(train_data, batch_size=batch)
        for i_epochs in range(epochs):    
            model.train()
            for x, y in train_loader:
                y = y.long()
                optmizer.zero_grad()
                output = model(x)
                loss = criterion(output, y)
                loss.backward()
                optmizer.step()

        model.eval()
        with torch.no_grad():
            x_test_tensor = Tensor(features[test_index]).float()
            y_true = targets[test_index]

            outputs = model(x_test_tensor)
            result = torch.argmax(outputs, dim=1).cpu().numpy()
         
            accuracy = accuracy_score(y_true, result)
            f1 = f1_score(y_true, result, average="macro")
            confusion = confusion_matrix(y_true, result)

            accuracy_scores.append(accuracy)
            conf_matrix.append(confusion)
            f1_scores.append(f1)
            
            print(f"Accuracy: {accuracy}")
            print(f"f1: {f1}")
            print(f"confusion: \n{confusion}")

    print(f"\nResultado modelo ----- Accuracy: {np.mean(accuracy_scores)}, f1: {np.mean(f1_scores)}")

In [4]:
dataset = load_iris()
features = dataset.data
targets = dataset.target

In [5]:
model = Sequential(
    Linear(4, 32),
    ReLU(),
    Linear(32,16),
    ReLU(),
    Linear(16, 3)
)

In [6]:
optmizer = Adam(model.parameters(), lr=0.001)
criterion = CrossEntropyLoss()
max_epochs = 500
batch = 8
skf = StratifiedKFold(n_splits=5, shuffle=True)

In [7]:
train_model_reg(features, targets, model, skf, batch, max_epochs, optmizer, criterion)

Treinando modelo ----- SKFold 1
Accuracy: 0.9666666666666667
f1: 0.9665831244778612
confusion: 
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]
Treinando modelo ----- SKFold 2
Accuracy: 1.0
f1: 1.0
confusion: 
[[10  0  0]
 [ 0 10  0]
 [ 0  0 10]]
Treinando modelo ----- SKFold 3
Accuracy: 0.9666666666666667
f1: 0.9665831244778612
confusion: 
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]
Treinando modelo ----- SKFold 4
Accuracy: 1.0
f1: 1.0
confusion: 
[[10  0  0]
 [ 0 10  0]
 [ 0  0 10]]
Treinando modelo ----- SKFold 5
Accuracy: 0.9
f1: 0.8976982097186701
confusion: 
[[10  0  0]
 [ 0 10  0]
 [ 0  3  7]]

Resultado modelo ----- Accuracy: 0.9666666666666668, f1: 0.9661728917348785
